Importamos las librerías de acuerdo a lo aprendido en clase

In [3]:
import pandas as pd
import os
import urllib.request, json, csv
import numpy as np
from tqdm import tqdm_notebook as tqdm

import requests

import os

import json

import pandas as pd

import csv

import datetime
import dateutil.parser
import unicodedata

import time
import requests

In [6]:
print(data["results"][0].keys())

dict_keys(['id', 'slug', 'name', 'released', 'tba', 'background_image', 'rating', 'rating_top', 'ratings', 'ratings_count', 'reviews_text_count', 'added', 'added_by_status', 'metacritic', 'playtime', 'suggestions_count', 'updated', 'user_game', 'reviews_count', 'saturated_color', 'dominant_color', 'platforms', 'parent_platforms', 'genres', 'stores', 'clip', 'tags', 'esrb_rating', 'short_screenshots'])


Definimos la función best_rated para ordenar los juegos en base a ranking (key 'metacritic'). Luego, utilizando esta función, enumeramos los primeros cinco resultados en un loop el diccionario "results" y pedimos que nos muestre el nombre del videojuego, el rating y el metacritic.

In [7]:
def best_rated(top_n=5):
    modified= requests.get(f"{url}",
                              params={"key":API_Key 
                                      ,"ordering":"-metacritic",
                                      "page_size": top_n
                                     })
    return modified.json()

In [8]:
print(f"Top 5 highest rated games")
trending= best_rated(top_n=5)

for i,game in enumerate(trending["results"],1):
    print(f"{i}.{game['name']} | Rating:{game['rating']}| Metacritic: {game['metacritic']}")

Top 5 highest rated games
1.The Legend of Zelda: Ocarina of Time | Rating:4.38| Metacritic: 99
2.Soulcalibur (1998) | Rating:0.0| Metacritic: 98
3.Soulcalibur | Rating:4.38| Metacritic: 98
4.Baldur's Gate III | Rating:4.44| Metacritic: 97
5.Metroid Prime | Rating:4.35| Metacritic: 97


B2: Creamos una nueva función (steam_only) la cual nos ayudará con el mismo setting que best_rated, pero aplicado a este contexto.

In [15]:
def steam_only(top_n=10):
    modified= requests.get(f"{url}",
                              params={"key":API_Key 
                                      ,"ordering":"-metacritic",
                                      "page_size": top_n
                                     })
    return modified.json()

Con esta función, iteramos para que (siguiendo un contador) se filtren juegos solo vendidos en Steam de una página de 50 juegos (ya que, en caso de poner 10, no necesariamente los 10 juegos se vendan en Steam). Así, se analizará los juegos filtrados hasta conseguir los 10 con mejor rating y dejar de iterar en ese punto de análisis.

In [16]:
print("Top 10 highest rated games on Steam")
trending_steam = steam_only(top_n=50)

iterar = 0
for game in trending_steam["results"]:
    tiendas = [s['store']['id'] for s in game['stores']]
    if 1 not in [s['store']['id'] for s in game['stores']]:
        continue
    iterar += 1
    print(f"{iterar}. {game['name']} | Rating: {game['rating']} | Metacritic: {game['metacritic']}")
    if iterar == 10:
        break

Top 10 highest rated games on Steam
1. Baldur's Gate III | Rating: 4.44 | Metacritic: 97
2. Half-Life 2: Update | Rating: 4.13 | Metacritic: 96
3. Half-Life | Rating: 4.38 | Metacritic: 96
4. Half-Life 2 | Rating: 4.48 | Metacritic: 96
5. BioShock | Rating: 4.36 | Metacritic: 96
6. Red Dead Redemption 2 | Rating: 4.59 | Metacritic: 96
7. Grand Theft Auto IV: Complete Edition | Rating: 4.57 | Metacritic: 95
8. Elden Ring | Rating: 4.38 | Metacritic: 95
9. XCOM 2: War of the Chosen | Rating: 4.4 | Metacritic: 95
10. Divinity: Original Sin 2 | Rating: 4.38 | Metacritic: 95


Part C - Comparisons

C1: Definimos otra función que establezca la página con una plataforma en específico (solo juegos que se puedan jugar en ciertas plataformas) en base a su id (platform_id).

In [29]:
def rating_plataforma(platform_id, top_n=5):
    modified = requests.get(f"{url}", params={
        "key": API_Key,
        "ordering": "-metacritic",
        "page_size": top_n,
        "platforms": platform_id
    })
    return modified.json()

Luego, utilizando dicha función, enumeramos los mejores cinco juegos en PC (platform_id=4) y PS5 (platform_id=187)

In [30]:
print("Top 5 PC (platform_id=4)")
pc = rating_plataforma(4)
for i, game in enumerate(pc["results"], 1):
    print(f"{i}. {game['name']} | Rating: {game['rating']} | Metacritic: {game['metacritic']}")

print()

print("Top 5 PS5 (platform_id=187)")
ps5 = rating_plataforma(187)
for i, game in enumerate(ps5["results"], 1):
    print(f"{i}. {game['name']} | Rating: {game['rating']} | Metacritic: {game['metacritic']}")

Top 5 PC (platform_id=4)
1. Baldur's Gate III | Rating: 4.44 | Metacritic: 97
2. Half-Life 2: Update | Rating: 4.13 | Metacritic: 96
3. Half-Life | Rating: 4.38 | Metacritic: 96
4. Red Dead Redemption 2 | Rating: 4.59 | Metacritic: 96
5. Half-Life 2 | Rating: 4.48 | Metacritic: 96

Top 5 PS5 (platform_id=187)
1. Baldur's Gate III | Rating: 4.44 | Metacritic: 97
2. Red Dead Redemption | Rating: 4.42 | Metacritic: 95
3. Elden Ring | Rating: 4.38 | Metacritic: 95
4. The Elder Scrolls V: Skyrim | Rating: 4.42 | Metacritic: 94
5. Quake | Rating: 4.25 | Metacritic: 94


Finalmente, se suma el metacritic de los cinco mejores juegos en PC y en PS5 para calcular el promedio de ambos y se puede ver que, en promedio, los juegos mejor valorados pertenecen a la plataforma PC.

In [18]:
promedio_pc = sum(g["metacritic"] for g in pc["results"]) / 5
promedio_ps5 = sum(g["metacritic"] for g in ps5["results"]) / 5

print(f"Promedio Metacritic PC: {promedio_pc}")
print(f"Promedio Metacritic PS5: {promedio_ps5}")

if promedio_pc > promedio_ps5:
    print("PC tiene los juegos mejor valorados")
else:
    print("PS5 tiene los juegos mejor valorados")


Promedio Metacritic PC: 96.2
Promedio Metacritic PS5: 95.0
PC tiene los juegos mejor valorados


C2: Primero, se crea un diccionario con tres juegos de nuestra elección, uniendo los juegos por medio de su ID dentro de la base de datos del API.

In [31]:
juegos_ids = {
    "GTA V": 3498,
    "The Witcher 3": 3328,
    "Red Dead Redemption 2": 28
}

Creamos una tabla de comparación por medio de un request a los settings de la página web. En base a dicha tabla, por medio de un loop, extraemos tanto los géneros como las plataformas correspondientes a los videojuegos.

Finalmente, la información extraída (plataformas y generos) se une con datos como el nombre y el metacritic para poder imprimir finalmente la tabla comparativa entre estos tres juegos.

In [32]:
tabla_comparacion = []

for nombre, game_id in juegos_ids.items():
    respuesta = requests.get(
        f"https://api.rawg.io/api/games/{game_id}",
        params={"key": API_Key}
    ).json()
    
    generos = ", ".join([g['name'] for g in respuesta['genres']])
    plataformas = ", ".join([p['platform']['name'] for p in respuesta['platforms']])
    
    tabla_comparacion.append({
        "Nombre": respuesta['name'],
        "Rating": respuesta['rating'],
        "Metacritic": respuesta['metacritic'],
        "Géneros": generos,
        "Plataformas": plataformas
    })

for i, juego in enumerate(tabla_comparacion, 1):
    print(f"{i}. {juego['Nombre']} | Rating: {juego['Rating']} | Metacritic: {juego['Metacritic']} | Géneros: {juego['Géneros']} | Plataformas: {juego['Plataformas']}")

1. Grand Theft Auto V | Rating: 4.47 | Metacritic: 92 | Géneros: Action | Plataformas: PlayStation 5, Xbox Series S/X, PlayStation 3, PC, PlayStation 4, Xbox 360, Xbox One
2. The Witcher 3: Wild Hunt | Rating: 4.64 | Metacritic: 92 | Géneros: Action, RPG | Plataformas: PlayStation 5, Xbox Series S/X, macOS, PlayStation 4, Nintendo Switch, PC, Xbox One
3. Red Dead Redemption 2 | Rating: 4.59 | Metacritic: 96 | Géneros: Action | Plataformas: PC, PlayStation 4, Xbox One


C3: Elegimos las categorías Shooter, Strategy, Racing y RPG (Role Playing Games), por lo que los indexamos en un diccionario junto con su id de juego dentro del API, para luego crear un loop en el cual, aparte de settear el espacio, calculamos el promedio del rating por medio de una operación simple, así como enumeramos también los juegos con mayor rating por medio de otro loop.

In [37]:
generos = {
    "Shooter": "shooter",
    "Strategy": "strategy", 
    "Racing": "racing",
    "RPG": "role-playing-games-rpg"
}

for nombre_genero, slug in generos.items():
    resultado = requests.get(f"{url}", params={
        "key": API_Key,
        "genres": slug,
        "ordering": "-rating",
        "page_size": 5
    }).json()
    
    juegos = resultado["results"]
    promedio = sum(g["rating"] for g in juegos) / len(juegos)
    print(f"\n{nombre_genero} — Promedio rating: {promedio:.2f}")
    for i, g in enumerate(juegos, 1):
        print(f"  {i}. {g['name']} | Rating: {g['rating']}")



Shooter — Promedio rating: 4.68
  1. Mass Effect Trilogy | Rating: 4.75
  2. Cyberpunk 2077: Phantom Liberty | Rating: 4.71
  3. The Last of Us Part I | Rating: 4.67
  4. Quake II: Enhanced Edition | Rating: 4.67
  5. Splatoon 2: Octo Expansion | Rating: 4.62

Strategy — Promedio rating: 4.71
  1. Super Robot Taisen: Original Generation | Rating: 4.83
  2. Victoria 2: Heart of Darkness | Rating: 4.71
  3. Shining Force III | Rating: 4.67
  4. Championship Manager Season 03/04 | Rating: 4.67
  5. Gemfire | Rating: 4.67

Racing — Promedio rating: 4.62
  1. Bike Baron | Rating: 4.67
  2. Ballistics | Rating: 4.67
  3. POD Gold | Rating: 4.62
  4. Midnight Club 3: DUB Edition Remix | Rating: 4.58
  5. Virtua Racing | Rating: 4.58

RPG — Promedio rating: 4.80
  1. The Elder Scrolls VI | Rating: 4.86
  2. The Witcher 3: Wild Hunt – Blood and Wine | Rating: 4.81
  3. The Witcher 3 Wild Hunt - Complete Edition | Rating: 4.8
  4. The Witcher 3: Wild Hunt – Hearts of Stone | Rating: 4.76
  5. M

C4: Para hacer la comparacion de los años elegidos (2019, 2020, 2021), creamos una lista para luego utilizarla en un loop que, al igual que en el anterior ejercicio,,calcule el promedio de metacritic en elaño y luego enumere los cinco juegos con mayor metacritic en cada año.

In [38]:
años = [2019, 2020, 2021]

for año in años:
    resultado = requests.get(f"{url}", params={
        "key": API_Key,
        "ordering": "-metacritic",
        "page_size": 5,
        "dates": f"{año}-01-01,{año}-12-31"
    }).json()
    
    juegos = resultado["results"]
    promedio = sum(g["metacritic"] for g in juegos if g["metacritic"]) / len(juegos)
    print(f"\n{año} — Promedio Metacritic: {promedio:.2f}")
    for i, g in enumerate(juegos, 1):
        print(f"  {i}. {g['name']} | Metacritic: {g['metacritic']}")



2019 — Promedio Metacritic: 90.80
  1. Disco Elysium | Metacritic: 91
  2. Resident Evil 2 | Metacritic: 91
  3. NieR:Automata Game of the YoRHa Edition | Metacritic: 91
  4. DRAGON QUEST XI S: Echoes of an Elusive Age - Definitive Edition | Metacritic: 91
  5. Sekiro: Shadows Die Twice | Metacritic: 90

2020 — Promedio Metacritic: 93.00
  1. Persona 5 Royal | Metacritic: 94
  2. Half-Life: Alyx | Metacritic: 93
  3. Hades | Metacritic: 93
  4. The Last of Us Part II | Metacritic: 93
  5. Demon's Souls (2020) | Metacritic: 92

2021 — Promedio Metacritic: 88.80
  1. Disco Elysium: Final Cut | Metacritic: 90
  2. Chicory: A Colorful Tale | Metacritic: 89
  3. Super Mario 3D World + Bowser’s Fury | Metacritic: 89
  4. Deathloop | Metacritic: 88
  5. Ratchet & Clank: Rift Apart | Metacritic: 88


C5: Por último, creamos un loop en el cual unimos en una lista los siguientes elementos: nombre del juego, rating, metacritic, fecha de lanzamiento y género principal. Luego, con ayuda de pandas, creamos un DataFrame para exportarlo en formato csv. 

In [40]:
resultado = requests.get(f"{url}", params={
    "key": API_Key,
    "ordering": "-metacritic",
    "page_size": 20
}).json()

top20 = []
for game in resultado["results"]:
    top20.append({
        "name": game["name"],
        "rating": game["rating"],
        "metacritic": game["metacritic"],
        "release_date": game["released"],
        "main_genre": game["genres"][0]["name"] if game["genres"] else "N/A"
    })

os.makedirs("api/output", exist_ok=True)

df_top20.to_csv("api/output/top20_rawg.csv", index=False)
print("Archivo guardado!")

df_top20.head()

Archivo guardado!


,name,rating,metacritic,release_date,main_genre
0,The Legend of Zelda: Ocarina of Time,4.38,99,1998-11-21,Action
1,Soulcalibur (1998),0.00,98,1998-07-30,Fighting
2,Soulcalibur,4.38,98,1998-07-30,Action
3,Baldur's Gate III,4.44,97,2023-08-03,Adventure
4,Metroid Prime,4.35,97,2002-11-17,Action


D - Insights & Conclusions

1. What was the most interesting thing you found in the data?

    Considerando mi poco conocimiento en videojuegos, me pareció sorprendente la terminología de los datos en este contexto (como la diferencia entre plataformas de juego y plataformas de compra). Además del incremento de gusto por los videojuegos al inicio de la pandemia de acuerdo con la Metacritic promedio de los 5 mejores juegos en 2020.

2. Which genre or platform surprised you the most and why?

    El género RPG, desconocía la existencia de ese tipo de juegos y saber que, dentro de los géneros que elegí, es el mejor rankeado me abrió los ojos a observar que se valora la aventura de vivir vidas alternas.

3. What other question would you ask this API if you had more time?

   Probablemente averiguaría cuales son los mejores juegos que puedan ser instalados en una MacBook y, en caso existieran comentarios, traería los comentarios al output para personalizar la experiencia del uso de APIs en este caso.

4. How many requests did you use in total? (call client.resumen_requests())

    Lo usé ocho veces, principalmente en loops de búsqueda de información para la parte C.